# trainer-class-skeleton composite — cx27: Trainer.training_step: forward → scalar loss → backward → opt.step

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `trainer-class-skeleton`, `backward-on-scalar-loss`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "trainer-class-skeleton"
DD_ATOM_IDS = ["trainer-class-skeleton", "backward-on-scalar-loss"]
DD_SUBTOPICS = ["Trainer: Trainer class skeleton", "PyTorch: backward()"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

The Trainer-class skeleton (ARENA convention) factors the training loop into methods so each step is independently testable. The central method is `training_step(batch)` which wires together forward + loss + backward + optimizer step. Two atoms compose:

1. **trainer-class-skeleton** — `__init__` stores `model`, `optimizer`, `loss_fn`, and a `step` counter; `training_step(batch)` is the per-batch hook.
2. **backward-on-scalar-loss** — inside `training_step`, the loss MUST be reduced to a scalar before `.backward()` is called. The scalar contract is what makes the rest of the training loop (`opt.step`, `opt.zero_grad`) well-defined.

**Anatomy of `training_step`.**
```python
def training_step(self, batch):
    x, y = batch
    logits = self.model(x)
    loss = self.loss_fn(logits, y)      # scalar.
    self.optimizer.zero_grad()
    loss.backward()                     # backward-on-scalar-loss.
    self.optimizer.step()
    self.step += 1
    return loss.item()
```

**Why test together.** This is the smallest unit that exercises the full forward-backward-step cycle inside the OO trainer wrapper.

### Composite Exercise — Trainer.training_step: forward → scalar loss → backward → opt.step

**Atoms exercised together**: `trainer-class-skeleton`, `backward-on-scalar-loss`

Implement `cx27_make_trainer()` which returns a `MiniTrainer` class.

Required structure:
- `MiniTrainer.__init__(self, model, optimizer, loss_fn)`:
  - Store `self.model`, `self.optimizer`, `self.loss_fn`.
  - `self.step = 0` (global step counter).
  - `self.history = []` (list of per-step `loss.item()` floats).
- `MiniTrainer.training_step(self, batch)`:
  - Unpack `x, y = batch`.
  - `logits = self.model(x)`.
  - `loss = self.loss_fn(logits, y)` — MUST be a scalar.
  - `self.optimizer.zero_grad()`.
  - `loss.backward()` (atom: backward-on-scalar-loss).
  - `self.optimizer.step()`.
  - `self.step += 1`.
  - `self.history.append(loss.item())`.
  - Return the scalar `loss` tensor.

The test verifies a regression task converges, the step counter increments by 1 per `training_step` call, the history list grows in sync, and gradients land on model params.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx27_make_trainer():
    """Return a MiniTrainer class whose training_step does forward+backward+opt.step."""
    raise NotImplementedError

def _test_cx27():
    MiniTrainer = cx27_make_trainer()

    # Tiny linear regression: y = 3x + 2.
    t.manual_seed(0)
    model = nn.Linear(1, 1)
    opt = t.optim.SGD(model.parameters(), lr=0.05)
    loss_fn = nn.MSELoss()
    trainer = MiniTrainer(model, opt, loss_fn)

    # Case A: initial state.
    assert trainer.model is model
    assert trainer.optimizer is opt
    assert trainer.loss_fn is loss_fn
    assert trainer.step == 0, f'step must start at 0; got {trainer.step}'
    assert trainer.history == [], 'history must start empty'

    # Case B: one training_step returns scalar loss + increments counters.
    x = t.randn(8, 1)
    y = 3.0 * x + 2.0
    loss = trainer.training_step((x, y))
    assert isinstance(loss, t.Tensor), f'training_step must return a Tensor; got {type(loss).__name__}'
    assert loss.ndim == 0, f'loss must be scalar; got shape {tuple(loss.shape)}'
    assert trainer.step == 1, f'step must be 1 after one call; got {trainer.step}'
    assert len(trainer.history) == 1
    assert isinstance(trainer.history[0], float)

    # Case C: backward populated grads on model params.
    for p in model.parameters():
        assert p.grad is not None, '.grad must be populated after training_step (backward fired)'

    # Case D: convergence — loss decreases across many training_step calls.
    for _ in range(100):
        trainer.training_step((x, y))
    assert trainer.step == 101, f'step should be 101 after 1+100 calls; got {trainer.step}'
    assert len(trainer.history) == 101
    assert trainer.history[-1] < trainer.history[0], (
        f'loss should decrease: first={trainer.history[0]:.4f} last={trainer.history[-1]:.4f}'
    )
    # Fit close to truth.
    assert abs(model.weight.item() - 3.0) < 0.3, f'weight should approach 3; got {model.weight.item():.3f}'
    assert abs(model.bias.item() - 2.0) < 0.3, f'bias should approach 2; got {model.bias.item():.3f}'
    _dd_passed.add('cx27')

_test_cx27()

<details><summary>Show solution — cx27</summary>

```python
def cx27_make_trainer():
    class MiniTrainer:
        def __init__(self, model, optimizer, loss_fn):
            # Atom A (trainer-class-skeleton): store deps + counters.
            self.model = model
            self.optimizer = optimizer
            self.loss_fn = loss_fn
            self.step = 0
            self.history = []

        def training_step(self, batch):
            x, y = batch
            logits = self.model(x)
            loss = self.loss_fn(logits, y)
            self.optimizer.zero_grad()
            # Atom B (backward-on-scalar-loss): scalar loss enables implicit grad seed.
            loss.backward()
            self.optimizer.step()
            self.step += 1
            self.history.append(loss.item())
            return loss

    return MiniTrainer
```

Returning the loss TENSOR (not `.item()`) lets the caller chain `loss.backward()` again if they want — but here we already called backward inside `training_step`. The history stores `.item()` (plain float) so it doesn't hold a reference to the graph and prevent GC.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx27'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx27',
        'subtopics': ["Trainer: Trainer class skeleton", "PyTorch: backward()"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()